In [1]:
from lib import epl 
from lib import OSN
from lib import readout
from lib import plots
import time
import numpy as np
import pickle 
import brian2 as b2
import brian2hears as b2h
import matplotlib.pyplot as plt
#import csv
import os.path
from os import path
%matplotlib inline
from preprocess import preprocessing as pre
from my_lib import my_readout

In [2]:
testSounds = ["00f0204f_nohash_2", "00970ce1_nohash_0", "004ae714_nohash_0", "004ae714_nohash_1", "0132a06d_nohash_0", "0132a06d_nohash_2", "0132a06d_nohash_4", "0135f3f2_nohash_1", "0137b3f4_nohash_0", "0137b3f4_nohash_1"]              
trainingSounds = ["00f0204f_nohash_2"]
scale = 15 

trainingIndex = [4000,6000, 5000,3000,3500]#adding the last 3
testingIndex = [4000, 4500, 8000, 7000, 6000, 5500, 8000, 6000, 4500, 7500]
range_f = 100
step_f = 10
cfN = 20 #channel frequency number
colors = ['y','k','b','y','k','b','y','k','b','y']

trainingAudio = pre.createTraining(trainingSounds, trainingIndex, cfN, range_f, step_f, scale)
testAudio = pre.createTesting(testSounds, cfN, testingIndex, colors, range_f, step_f, scale)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/micha/A-Neuromorphic-App-to-Keyword-Recog-main/A-Neuromorphic-App-to-Keyword-Recog-main/yes_sounds/00f0204f_nohash_2.wav'

In [ ]:
from lib import epl 

nOdors = len(trainingOdors); 
nTestPerOdor = len(testOdors)/nOdors;  
print("Number of odors to train = " + str(len(trainingOdors))); 
print("Number of odors to test = " + str(len(testOdors))); 

#Network initialization
nMCs = len(trainingOdors[0]); 
GCsPerNeurogenesis = 5; 
nGCs = nMCs*GCsPerNeurogenesis*nOdors;     #every MC has 5 GCs per odor  
epl = epl.EPL(nMCs, nGCs, GCsPerNeurogenesis); 

#Sniff
def sniff(odor, learn_flag=0, nGammaPerOdor=5, gPeriod=40):
    sensorInput = OSN.OSN_encoding(odor); 
    for j in range(0, nGammaPerOdor):
        for k in range(0, gPeriod): 
            epl.update(sensorInput, learn_flag=learn_flag);
            pass; 
    epl.reset(); 

#Training
t1 = time.time(); 
for i in range(0, len(trainingOdors)):
    print("Training odor " + str(i+1)) 
    sniff(trainingOdors[i], learn_flag=1,nGammaPerOdor = 10); 
    # changed nGammaPerOdor from 5 to 10
    epl.GClayer.invokeNeurogenesis();
    sniff(trainingOdors[i], learn_flag=0);

#Testing
for i in range(0, len(testOdors)):
    sniff(testOdors[i], learn_flag=0); 
    if(i%10==0 and i!=0):
        print(str(i) + " odors tested"); 
t2 = time.time(); 
print("Simulation Duration = " + str(t2-t1) + "s")

#Readout
# has summary of a lot of the results
# find structure of sMatrix and summarize the performance for a particular run

sMatrix = readout.readout(epl.gammaCode, nOdors, nTestPerOdor)  
# f=open('sMatrix.txt','a',fmt='%i',delimiter=', ')
# np.savetxt(f,sMatrix)
# f.close()

In [ ]:
len(epl.gammaCode), len(epl.gammaCode[0])

In [ ]:
from my_lib import my_plots
from my_lib import my_readout
sMatrix = my_readout.readout(epl.gammaCode, nOdors, nTestPerOdor, nGammaPerLearning = 10)#, nGammaPerOdor_training = nGammaPerOdor_training)  
plotTitles= ['Sound 1', 'Sound 2', 'Sound 3', 'Sound 4', 'Sound 5']

my_plots.learningBarFigure(sMatrix, plotTitles, nGamma = 10)

my_plots.rasterSimilarityPlot(
    epl.gammaCode,
    sMatrix,
    plotTitles,
    nMCs=nMCs,
    nGamma=5,
    nRows=5,
    nCols=4
)

In [ ]:
plotFigure4d_audio(epl.gammaCode, sMatrix)

In [ ]:
len(sMatrix), len(sMatrix[0])

In [ ]:
def plotFigure4d_audio(gammaCode, sMatrix):
    fig, axs = plt.subplots(figsize=(18, 9))
    fig.suptitle("Figure 4d - Spectogram-sliced Audio", fontsize=16)

    # Use digits you have
    soundNum = ['Sound 1', 'Sound 2', 'Sound 3', 'Sound 4', 'Sound 5']
    nOdors = len(soundNum)
    plotTitles = [f"Sound {d}" for d in soundNum]

    # Adjust plotting order if needed
    rasterOrderOdors = list(range(nOdors))
    sampleNumbers = [0] * nOdors  # default

    nMCs = len(gammaCode[0])
    nGamma = 5
    nTestPerOdor = int(len(sMatrix) / (nOdors * nGamma))

    plt.subplots_adjust(hspace=0.45, wspace=0.25)
    fsize = 12
    fsize2 = 16

    for i in range(nOdors):
        # Raster plot
        plt.subplot(3, 4, 2 * i + 1)
        odorID = rasterOrderOdors[i]

        sniffID = nOdors * 2 + nTestPerOdor * odorID + sampleNumbers[odorID]

        for j in range(nGamma):
            gammaID = int(nGamma * sniffID + j)
            for k in range(nMCs):
                if gammaCode[gammaID][k] != 0:
                    spikeTime = 20 - gammaCode[gammaID][k] + 40 * j
                    plt.scatter(spikeTime, k, color='k', s=2)
        plt.title(plotTitles[i], fontsize=fsize2)
        plt.yticks([0, 20, 40, 60], fontsize=fsize)
        if i >= (nOdors - 2):
            plt.xticks([0, 40, 80, 120, 160, 200], fontsize=fsize)
            plt.xlabel('Timesteps', fontsize=fsize2)
            plt.ylabel('MC Index', fontsize=fsize2)
        else:
            plt.xticks([])

        # Similarity plot
        plt.subplot(3, 4, 2 * i + 2)
        bar_y = []
        gammaID = odorID * nTestPerOdor * nGamma + sampleNumbers[odorID] * nGamma
        for j in range(5):
            print(int(gammaID + j))
            print(sMatrix[int(gammaID + j)][odorID])
            bar_y.append(sMatrix[int(gammaID + j)][odorID])
            
            
        plt.bar([40, 80, 120, 160, 200], bar_y, color='black', width=10.0)
        plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], fontsize=fsize)
        plt.ylim([0.0, 1.02])
        plt.xlim([0, 240])
        if i >= (nOdors - 2):
            plt.xticks([40, 80, 120, 160, 200], [1, 2, 3, 4, 5], fontsize=fsize)
            plt.xlabel('Gamma Cycles', fontsize=fsize2)
            plt.ylabel('Similarity', fontsize=fsize2)
        else:
            plt.xticks([])
    plt.show()
def plotFigure4b_audio(sMatrix): 
    bar1_x, bar2_x, bar3_x, bar4_x, bar5_x = [], [], [], [], [];      
    bar1_y, bar2_y, bar3_y, bar4_y, bar5_y = [], [], [], [], [];      
    nGamma = 5; 
    testOdorID = 0;              #sniff ID of test odor 
    for i in range(0, nGamma):
        bar1_y.append(sMatrix[testOdorID+i][0]);
        #bar2_y.append(sMatrix[testOdorID+i][4]);
        #bar3_y.append(sMatrix[testOdorID+i][5]);
        bar4_y.append(sMatrix[testOdorID+i][1]);
        #bar5_y.append(sMatrix[testOdorID+i][2]);
    w = 0.15        #width of bars
    fsize = 14;
    xticks = []
    xtick_labels = []
    for i in range(0, nGamma):    
        bar1_x.append(i-2*w);
        bar2_x.append(i-w);
        bar3_x.append(i);
        bar4_x.append(i+w); 
        bar5_x.append(i+2*w);     
        xticks.append(i)
        xtick_labels.append(str(i+1))
    plt.figure(2, figsize=(10, 5))
    fig = plt.subplot(111)
    opacity = 0.5; 
    bar1 = plt.bar(bar1_x, bar1_y, width = w, color = 'blue', alpha = opacity, align='center')
    #bar2 = plt.bar(bar2_x, bar2_y, width = w, color = 'violet', alpha = opacity, align='center')
    #bar3 = plt.bar(bar3_x, bar3_y, width = w, color = 'red', alpha = opacity, align='center')
    bar4 = plt.bar(bar4_x, bar4_y, width = w, color = 'orange', alpha=opacity, align='center')
    #bar5 = plt.bar(bar5_x, bar5_y, width = w, color = 'green', alpha=opacity, align='center')
    plt.xlim(-1, len(bar1_x)+0.2)
    plt.ylim(0, 1.02)
    plt.xticks(xticks, xtick_labels, fontsize=fsize)
    plt.yticks(fontsize=fsize); 
    fig.legend( (bar1,  bar4), ('0', '2','4','5','7','9'), loc = 'upper left', fontsize=fsize)
    plt.ylabel('Similarity', fontsize=fsize); 
    plt.xlabel('Gamma Cycles', fontsize=fsize); 
    plt.title("Figure 4b")
    plt.show()

In [ ]:
#Plots
print(len(sMatrix), len(sMatrix[0]))  # Check the shape of the similarity matrix
plotFigure4b_audio(sMatrix); 
plotFigure4d_audio(epl.gammaCode, sMatrix); 